 # PolicyProof AI — Stage 3: AlignTune QLoRA T4

Research-only integration test. Run in a **new Google Colab T4 notebook**. Use your downloaded CuratorKIT ZIP, which contains only fictional training rows. The previous 48-case benchmark must never enter training. **Installation and GPU training default to OFF.** This is not a verified model improvement. Tiny Aya's CC-BY-NC license applies.

## 1. Check T4 and pull versioned project code.

In [1]:
import torch, subprocess, pathlib, os, sys, json, time
assert torch.cuda.is_available(), "Select a T4 GPU under Runtime settings and connect."
print("GPU:", torch.cuda.get_device_name(0))
ROOT=pathlib.Path("/content/PolicyProof-AI")
if not ROOT.exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/betterlifestyle04-commits/PolicyProof-AI.git",str(ROOT)],check=True)
else:
    subprocess.run(["git","-C",str(ROOT),"pull","--ff-only"],check=True)
os.chdir(ROOT)
sys.path.insert(0,str(ROOT))
REV=subprocess.check_output(["git","rev-parse","HEAD"],text=True).strip()
print("Code revision:",REV)


GPU: Tesla T4
Code revision: 74c4ce6189fae058f49dc3e90037d695715a9adc


## 2. Upload the CuratorKIT training ZIP and verify its integrity before extraction. Only allowlisted files are extracted.

In [2]:
from google.colab import files
from zipfile import ZipFile
from scripts.verify_curator_archive import check_zip
uploaded=files.upload()
assert len(uploaded)==1, "Upload only the policyproof_curatorkit_train_v1.zip"
zip_name=next(iter(uploaded))
assert zip_name.endswith(".zip")
report=check_zip(zip_name)
assert report["status"]=="pass" and report["row_count"]==78
print(json.dumps(report,indent=2,ensure_ascii=False))
CURATED=ROOT/"artifacts"/"verified_curator_train"
CURATED.mkdir(parents=True,exist_ok=True)
allowed={"sft_alpaca.jsonl","sft_sharegpt.jsonl","manifest.json",
         "lexsi_provenance.json","checksums.txt","README.md","dataset_card.md","rejected.jsonl"}
with ZipFile(zip_name) as z:
    for name in sorted(allowed.intersection(z.namelist())):
        (CURATED/name).write_bytes(z.read(name))


Saving policyproof_curatorkit_train_v1.zip to policyproof_curatorkit_train_v1.zip
{
  "status": "pass",
  "tool_version": "0.1.22",
  "row_count": 78,
  "languages": {
    "English": 26,
    "Norwegian": 26,
    "Arabic": 26
  },
  "abstentions": 6,
  "policy_labels": {
    "ABSTAIN": 6,
    "POL-001": 6,
    "POL-002": 6,
    "POL-003": 6,
    "POL-004": 6,
    "POL-005": 6,
    "POL-006": 6,
    "POL-007": 6,
    "POL-008": 6,
    "POL-009": 6,
    "POL-010": 6,
    "POL-011": 6,
    "POL-012": 6
  },
  "verified_files": [
    "lexsi_provenance.json",
    "manifest.json",
    "rejected.jsonl",
    "sft_alpaca.jsonl",
    "sft_sharegpt.jsonl"
  ],
  "source_files_logged": [],
  "generation_or_semantic_validation": false,
  "human_reviewed": false,
  "limits": [
    "All data are fictional and template-generated",
    "Exact duplicates only: near-duplicates and semantic contradictions unchecked",
    "Not proof of real-world safety or multilingual quality",
    "Only training IDs permi

## 3. Authenticate Hugging Face and record the exact model revision

Use your approved Hugging Face account and a read-only token via the login widget if necessary. Do not paste credentials into a notebook cell. This step does not download model weights.

In [3]:
from huggingface_hub import whoami, notebook_login, model_info
try:
    hf_account = whoami()
except Exception:
    notebook_login()
    hf_account = whoami()
MODEL_ID = "CohereLabs/tiny-aya-global"
MODEL_SHA = model_info(MODEL_ID).sha
assert isinstance(MODEL_SHA, str) and len(MODEL_SHA) >= 12
print("HF account:", hf_account["name"])
print("Approved model:", MODEL_ID)
print("Model commit SHA:", MODEL_SHA)


HF account: shalnark01
Approved model: CohereLabs/tiny-aya-global
Model commit SHA: af89d219b53ed9b13b8a4645f9c8028973510324


## 4. Restore missing training packages in this Colab runtime

**A new Colab runtime does not retain pip-installed libraries or Python objects.** The previous successful setup used AlignTune **0.1.28**, TRL **1.7.1**, and BitsAndBytes **0.50.2**. This cell installs those exact versions only when missing and leaves the already-installed PyTorch in place. It checks the actual training imports and T4 CUDA independently from unrelated global `pip check` warnings (such as missing Jedi).

Running this cell **does not** restore `trainer` or previously loaded Tiny Aya weights. If the runtime was reset, repeat the verified archive and model-loading steps before training. No GPU training starts in this cell.

In [4]:
import subprocess, sys, importlib
from importlib.metadata import version, PackageNotFoundError

REQUIRED_TRAINING_PACKAGES = {
    "aligntune": "aligntune==0.1.28",
    "trl": "trl==1.7.1",
    "bitsandbytes": "bitsandbytes==0.50.2",
}
missing = []
for pkg, spec in REQUIRED_TRAINING_PACKAGES.items():
    try:
        print(f"{pkg}: {version(pkg)}")
    except PackageNotFoundError:
        missing.append(pkg)

print("Missing training packages:", missing)
if missing:
    print("Installing the previously verified compatible package set.")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--prefer-binary",
         *REQUIRED_TRAINING_PACKAGES.values(), "jedi"],
        check=True
    )
    importlib.invalidate_caches()
else:
    print("Training packages already installed; skipping reinstall.")

print("\nDEPENDENCY VERSION CHECK:")
for pkg in ["aligntune", "trl", "bitsandbytes", "transformers",
            "datasets", "huggingface-hub", "peft", "torch"]:
    try:
        print(f"{pkg}: {version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg}: NOT INSTALLED")

# Global pip check can flag unrelated preinstalled Colab package conflicts.
pipcheck = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True, text=True
)
print("\nGlobal pip check code:", pipcheck.returncode)
print((pipcheck.stdout or pipcheck.stderr)[-2500:])

smoke_src = """
import torch
import bitsandbytes
from trl import SFTTrainer
from peft import LoraConfig
from aligntune.core.backend_factory import create_sft_trainer
assert callable(create_sft_trainer)
assert torch.cuda.is_available(), 'CUDA GPU unavailable. Select T4 runtime.'
torch.empty(1, device='cuda')
print('CORE_IMPORTS_AND_CUDA_OK:', torch.cuda.get_device_name(0))
"""
smoke = subprocess.run(
    [sys.executable, "-c", smoke_src],
    capture_output=True, text=True, timeout=180
)
print("\nSMOKE TEST EXIT CODE:", smoke.returncode)
print("stdout:", smoke.stdout[-3500:])
print("stderr:", smoke.stderr[-4000:])
TRAINING_ENV_READY = smoke.returncode == 0 and "CORE_IMPORTS_AND_CUDA_OK" in smoke.stdout
print("TRAINING_ENV_READY:", TRAINING_ENV_READY)

print("\nLive Python objects in this Colab notebook:")
print("trainer exists:", "trainer" in globals())
print("CURATED exists:", "CURATED" in globals())
if "trainer" not in globals():
    print("Fresh/cleared session: library installation is not enough. "
          "Run model and dataset setup before attempting training.")
if not TRAINING_ENV_READY:
    raise RuntimeError("Training libraries or GPU validation failed. Do not run training.")


Missing training packages: ['aligntune', 'trl', 'bitsandbytes']
Installing the previously verified compatible package set.

DEPENDENCY VERSION CHECK:
aligntune: 0.1.28
trl: 1.7.1
bitsandbytes: 0.50.2
transformers: 5.22.0
datasets: 4.8.5
huggingface-hub: 1.33.0
peft: 0.21.2
torch: 2.11.0+cu130

Global pip check code: 1
cuml-cu13 26.6.0 has requirement numba<0.65.0,>=0.60.0, but you have numba 0.65.0.
google-adk 2.7.1 has requirement opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2.
google-adk 2.7.1 has requirement opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2.
cudf-cu13 26.6.0 has requirement numba<0.65.0,>=0.60.0, but you have numba 0.65.0.


SMOKE TEST EXIT CODE: 0
stdout: [aligntune] 2026-11-04 20:47:37 WARNING aligntune.core.rl.registries:54 - Could not initialize reward 'rubric' from rewards registry: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY`

## 4b. Fresh Colab runtime? Restore the model in one cell

If Step 4 reported **`trainer still loaded: False`**, run this **one** recovery cell. It verifies any existing curated data or prompts you to upload your previously downloaded `policyproof_curatorkit_train_v1.zip`, checks its original SHA256, signs into Hugging Face only when necessary, and loads the same 4-bit Tiny Aya/LoRA model. It validates all 78 tokenization examples without starting training. **After it reports `RESTORE COMPLETE`, skip Steps 5 and 6** to avoid loading another copy of the model. Then use the guarded training fallback as a separate action.

In [10]:
# Safe one-call restoration after Colab reset; no training is executed.
import pathlib, subprocess, sys, os, json
ROOT = pathlib.Path("/content/PolicyProof-AI")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1",
       "https://github.com/betterlifestyle04-commits/PolicyProof-AI.git",
       str(ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

assert "trainer" not in globals() or getattr(trainer, "model", None) is None, (
    "A model is already loaded. Do not allocate a duplicate."
)
from scripts.restore_colab_session import restore_interactive
trainer, MODEL_SHA, restore_report = restore_interactive(ROOT)
print(json.dumps(restore_report, indent=2))


AssertionError: A model is already loaded. Do not allocate a duplicate.

## 5. Dry-run the exact AlignTune training configuration (no model weights)

After validating CuratorKIT ZIP extraction and successfully importing AlignTune, run this cell to validate the 78 training records, ensure no held-out policy IDs are present, instantiate AlignTune's **TRL trainer wrapper only**, and check the LoRA + 4-bit settings. This does **not** download a model or train.

In [6]:
import subprocess, sys, json
from pathlib import Path
train_path = CURATED / "sft_alpaca.jsonl"
assert train_path.exists(), "Run archive upload/validation first"
config_only=subprocess.run(
    [sys.executable, "-m", "scripts.check_align_config",
     "--curated-dir", str(CURATED)],
    capture_output=True, text=True, timeout=180,
)
print("AlignTune CONFIG-ONLY CHECK EXIT:", config_only.returncode)
print(config_only.stdout[-8500:])
print(config_only.stderr[-5000:])
CONFIG_VALIDATED = config_only.returncode == 0 and "config_only_valid" in config_only.stdout
print("CONFIG_VALIDATED:", CONFIG_VALIDATED)
if not CONFIG_VALIDATED:
    print("STOP. Share the error output; do not start model training.")


AlignTune CONFIG-ONLY CHECK EXIT: 0
[aligntune] 2026-11-04 20:50:26 WARNING aligntune.core.rl.registries:54 - Could not initialize reward 'rubric' from rewards registry: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.
{
  "status": "config_only_valid",
  "data": {
    "row_count": 78,
    "languages": {
      "English": 26,
      "Norwegian": 26,
      "Arabic": 26
    },
    "abstention_count": 6,
    "dataset_sha256": "4b14eb262b52eaba00794f070a4e69ae543bd2d13d060d2d06e4832c1a84294c"
  },
  "trainer": {
    "trainer_class": "TRLSFTTrainer",
    "backend": "trl",
    "model": "CohereLabs/tiny-aya-global",
    "config_name": "sft_alpaca",
    "training_steps": 10,
    "micro_batch": 1,
    "max_seq_length": 1024,
    "peft_enabled": true,
    "lora_rank": 8,
    "quantized_4bit": true,
    "loaded_model_weights": false,
    "trained_model": false,
    "model_specific_compatibili

## 6. Initialize the quantized model and LoRA adapter — controlled load, NO training

This is the next compatibility check. It may download model weights and consume several GB of T4 memory. It calls `setup_model()` and `setup_data()` but **does not** call `train()`. We verify actual 4-bit modules, positive LoRA trainable parameters, and 78 training rows. If the T4 runs out of memory, don't start training; share the error.

In [7]:
RUN_MODEL_LOAD = False  # Change to True after configuration-only preflight passes
MODEL_LOAD_VALIDATED = False
if RUN_MODEL_LOAD:
    assert CONFIG_VALIDATED, "Run Stage 5 and confirm CONFIG_VALIDATED=True"
    import torch, bitsandbytes as bnb
    from scripts.check_align_config import make_training_wrapper
    assert torch.cuda.is_available()
    if "trainer" in globals() and getattr(trainer, "model", None) is not None:
        raise RuntimeError("A model is already loaded. Do not load a second copy.")
    trainer = make_training_wrapper(CURATED)
    trainer.setup_model()  # Loads approved model + applies 4-bit LoRA; no training.
    model = trainer.model
    four_bit_layers = sum(isinstance(m, bnb.nn.Linear4bit) for m in model.modules())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print("4-bit linear layers:", four_bit_layers)
    print("Trainable LoRA parameters:", trainable, "of", total)
    print("GPU allocated GiB:", round(torch.cuda.memory_allocated()/1024**3, 2))
    assert four_bit_layers > 0, "4-bit quantization failed"
    assert 0 < trainable < total, "No trainable LoRA adapters detected"
    trainer.setup_data()   # Loads training data only, not 48-case holdout.
    train_rows = len(trainer.train_dataset)
    print("Prepared training records:", train_rows)
    assert train_rows > 0
    MODEL_LOAD_VALIDATED = True
    print("MODEL LOAD AND DATA SETUP OK — no training performed")
else:
    print("Model load disabled. Turn it on only after Stage 5 validates.")


Model load disabled. Turn it on only after Stage 5 validates.


## 7. One-call, guarded T4 recovery and ten-step training

AlignTune 0.1.28 stores stale precision flags in `config.train.extra_params`. Those can override `model.precision` and re-enable FP16 AMP. The helper in the GitHub repo synchronizes both precision sources, casts only the LoRA adapters to FP32 if necessary, validates that the actual TRL SFTConfig has `fp16=False` and `bf16=False`, trains **once**, checks exactly ten optimizer steps, verifies the saved adapter, and creates a ZIP.

**Training defaults OFF**. Turn `RUN_TRAINING=True` only when your curated 78-row dataset is loaded on a T4 and you're ready to run. A successful adapter is not proof of improved quality. If training fails the helper creates a failure report and does not retry automatically.

In [8]:
RUN_TRAINING = False
if RUN_TRAINING:
    from scripts.complete_t4_training import recover_train_save
    from google.colab import files
    archive = recover_train_save(
        trainer,
        project_dir=str(ROOT),
        model_revision=globals().get("MODEL_SHA"),
    )
    files.download(str(archive))
else:
    print("Training OFF; no optimizer steps executed.")


Training OFF; no optimizer steps executed.


## 8. T4 recovery fallback — Direct PyTorch optimizer (no TRL/AlignTune trainer)

**Use only if the AlignTune/TRL trainer repeatedly fails with BF16 GradScaler on the T4.** TRL 1.7.1 automatically recasts LoRA parameters to BF16 on quantized models during initialization. This fallback uses the already-loaded AlignTune model and CuratorKIT data but performs 10 optimizer steps with PyTorch directly, avoiding a TRL constructor and automatic GradScaler. This is a **technical integration workaround** and must be labeled as a *PyTorch fallback*, **not** proof that the AlignTune trainer executed successfully. It also does not demonstrate improved accuracy or safety.

No new model is downloaded, and no eval/test data enter training. It validates the exact 78-example dataset SHA, 4-bit module count, and 3,833,856 trainable LoRA parameters, then saves the PEFT adapter and hashes. If a prior fallback receipt exists, it refuses to overwrite. Training remains OFF by default.

In [9]:
RUN_TORCH_FALLBACK = False  # Explicitly opt in when ready
if RUN_TORCH_FALLBACK:
    from scripts.t4_torch_fallback import train_existing_t4
    from google.colab import files
    archive = train_existing_t4(
        trainer,
        root="/content/PolicyProof-AI",
        model_sha=globals().get("MODEL_SHA"),
    )
    files.download(str(archive))
else:
    print("Fallback optimizer disabled; no GPU training performed.")


Fallback optimizer disabled; no GPU training performed.


## Next stage

If the install fails, share the traceback without credentials. If it works, download the adapter plus run_info evidence. Then build a fresh uninspected multilingual benchmark and verify the tuned model independently. Official docs: https://aligntune.lexsi.ai/user-guide/sft/ .

## Optional: genuine 40-step QLoRA training run — separate from the 10-step smoke

Use this **instead of** the earlier 10-step AlignTune or PyTorch smoke training cells, with a freshly restored, never-trained 4-bit Tiny Aya model and the exact 78-row CuratorKIT training archive. It deliberately uses a distinct output folder and ZIP.

The first 66 deterministically selected **training-source** rows receive optimizer updates; the remaining 12 **training-source** rows are used only for a small held-out loss check after steps 10, 20, 30 and 40. None of the 48-case development benchmark, 60-case final candidate, or 72-case adversarial challenge is used for training or validation. This does NOT prove improved safety.

This path uses **direct PyTorch over an AlignTune-loaded PEFT model**, not a successful AlignTune trainer run. It avoids the TRL FP16/BF16 GradScaler incompatibility observed previously. It requires actual GPU access and a full pinned 40-hex base revision. Training is OFF by default. On success, save and securely download the adapter ZIP and training receipt; no benchmark result is claimed.

In [ ]:
RUN_EXTENDED_TRAINING = False  # Explicit opt-in only; starts CUDA optimizer steps
if RUN_EXTENDED_TRAINING:
    assert not globals().get("RUN_TRAINING", False), "Do not run smoke and extended training in one session"
    assert not globals().get("RUN_TORCH_FALLBACK", False), "Do not run both training modes in one session"
    assert "trainer" in globals() and getattr(trainer, "model", None) is not None
    assert len(MODEL_SHA) == 40, "Pin the exact Hugging Face base revision"
    from scripts.t4_torch_fallback import training_plan, train_existing_t4
    print("TRAINING SCHEDULE:", {k: v for k, v in training_plan(steps=40).items()
          if k not in ("train_indices", "validation_indices")})
    archive = train_existing_t4(
        trainer, root=str(ROOT), model_sha=MODEL_SHA, optimizer_steps=40,
    )
    from google.colab import files
    files.download(str(archive))
else:
    print("Extended training OFF: no GPU optimizer steps executed.")
